# DataMind Survey: Run Experiments 1-3 on Colab Pro

Use this notebook for the feasible project scope: Exp. 1, Exp. 2, and Exp. 3. It avoids OpenAI API quota by using a local deterministic judge and serves Qwen2.5-7B with vLLM on the Colab GPU.

Before running: in Colab, click `Runtime` -> `Change runtime type` -> choose `GPU`. Prefer `A100` or `L4`; `T4` can work for smaller `N_SAMPLES` but may be slow.

In [6]:
# Configuration
REPO_URL = "https://github.com/Razim021/datamind-survey.git"
BRANCH = "main"

N_SAMPLES = 5
MODEL_NAME = "Qwen2.5-7B-Instruct"
HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"
API_PORT = 8000
MAX_ROUNDS = 6

print("Configured for", MODEL_NAME, "with", N_SAMPLES, "samples")

Configured for Qwen2.5-7B-Instruct with 50 samples


In [7]:
# Check GPU
!nvidia-smi

Sat Apr 25 15:22:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
# Clone project and install dependencies
import os
from pathlib import Path

PROJECT_DIR = Path("/content/datamind-survey")
if not PROJECT_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} /content/datamind-survey

%cd /content/datamind-survey

# Upstream dependency used by the wrapper scripts.
if not Path("Datamind-main").exists():
    !git clone https://github.com/zjunlp/DataMind Datamind-main

!pip install -q -r requirements.txt
!pip install -q vllm
!python download_data.py

os.environ["DATAMIND_ROOT"] = "/content/datamind-survey/Datamind-main"
os.environ["DATA_DIR"] = "/content/datamind-survey/data"
os.environ["JUDGE_BACKEND"] = "local"
print("Setup complete")

Cloning into '/content/datamind-survey'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/datamind-survey'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
python3: can't open file '/content/download_data.py': [Errno 2] No such file or directory
Setup complete


In [9]:
# Sanity-check data loaders
!python - <<'PY'
from experiments.utils.data_loader import load_qrdata, load_discoverybench
print('QRData samples:', len(load_qrdata('data')))
print('DiscoveryBench samples:', len(load_discoverybench('data')))
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


ModuleNotFoundError: No module named 'experiments'

In [ ]:
# Start vLLM server in the background
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path('/content/vllm.log')
server_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', HF_MODEL,
    '--served-model-name', MODEL_NAME,
    '--port', str(API_PORT),
    '--dtype', 'float16',
    '--max-model-len', '4096',
]

server_log = open(log_path, 'w')
vllm_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
print('Started vLLM process:', vllm_proc.pid)
print('Log:', log_path)

for i in range(90):
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/v1/models', timeout=2) as response:
            print('vLLM is ready')
            break
    except Exception:
        time.sleep(5)
else:
    print('vLLM did not become ready. Last log lines:')
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-40:]))

In [ ]:
# Experiment 1: data-comprehension metadata ablation
%cd /content/datamind-survey/experiments
!python run_exp1_comprehension.py \
  --model_name {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --sub_experiment info \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --max_rounds {MAX_ROUNDS} \
  --judge_backend local \
  --output_dir results/exp1_colab

In [ ]:
# Experiment 2: code-generation and error analysis
!python run_exp2_code_analysis.py \
  --mode evaluate \
  --models {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --max_rounds {MAX_ROUNDS} \
  --judge_backend local \
  --output_dir results/exp2_colab

In [ ]:
# Experiment 2b: local heuristic error categorization
!python run_exp2_code_analysis.py \
  --mode categorize \
  --results_dir results/exp2_colab \
  --n_errors {N_SAMPLES} \
  --judge_backend local \
  --output_dir results/exp2_colab

In [ ]:
# Experiment 3: turn-budget study
!python run_exp3_turn_budget.py \
  --model_name {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --turn_budgets 2,4,6 \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --judge_backend local \
  --output_dir results/exp3_colab

In [ ]:
# Show result summaries
!find results -maxdepth 3 -type f \( -name 'summary.json' -o -name 'error_categories.json' \) -print -exec python -m json.tool {} \;

In [ ]:
# Zip results for download
%cd /content/datamind-survey/experiments
!zip -r /content/datamind_first3_results.zip results
print('Download /content/datamind_first3_results.zip from the Colab file browser.')